# Week 13 — Neural Network Surrogate Training (FINAL ROUND)

Same architecture family as W4-W12: MLP with H in {16, 32}, four regularisation variants, 5-fold CV across 8 configs.

Re-trains on the latest data including W12 portal returns (22/22/27/42/32/32/42/52 pts).

W12 produced 6 new bests (F3-F8). F1 envelope shot landed negative (sign finer than the data resolves); F2 drew below its bank.

In [1]:
import sys, json
from pathlib import Path
import numpy as np
sys.path.append('../src')
import nn_models as nm

MODELS_DIR = Path('../models/week_13')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

WIDTHS = [16, 32]
VARIANTS = list(nm.VARIANTS)

def load(n):
    X = np.load(f'../data/function_{n}/initial_inputs.npy')
    Y = np.load(f'../data/function_{n}/initial_outputs.npy')
    return X, Y


In [2]:
def search_and_save(n, verbose=True):
    X, Y = load(n)
    baseline = float(Y.std())
    results = []
    for H in WIDTHS:
        for v in VARIANTS:
            rmse = nm.cv_rmse(X, Y, v, hidden=H, n_splits=5, seed=0)
            results.append((rmse, H, v))
    results.sort(key=lambda r: r[0])
    best_rmse, best_H, best_v = results[0]
    improvement = (baseline - best_rmse) / baseline * 100
    beat_baseline = best_rmse < baseline

    if verbose:
        print(f'F{n}: {X.shape[0]} pts, {X.shape[1]}D, baseline RMSE = {baseline:.4f}')
        print(f'  All configs (sorted):')
        for r, H, v in results:
            mark = ' ← BEST' if (r, H, v) == (best_rmse, best_H, best_v) else ''
            print(f'    H={H:3d}  {v:10s}  CV RMSE = {r:.4f}{mark}')
        status = '✓ beats baseline' if beat_baseline else '✗ WORSE than baseline'
        print(f'  → best: H={best_H}, {best_v}, RMSE={best_rmse:.4f} ({improvement:+.1f}% vs baseline) {status}')

    # Fit final model on all data
    models, meta = nm.fit_final(X, Y, best_v, best_H)
    meta['cv_rmse'] = best_rmse
    meta['baseline_rmse'] = baseline
    meta['beats_baseline'] = beat_baseline
    meta['all_configs'] = [{'hidden': H, 'variant': v, 'rmse': r} for r, H, v in results]

    # Gradient at current best point
    x_best = X[Y.argmax()]
    grad = nm.gradient_at(models, meta, x_best)
    meta['gradient_at_best'] = grad.tolist()
    meta['x_best'] = x_best.tolist()
    meta['y_best'] = float(Y.max())

    if verbose:
        print(f'  Gradient dY/dx at best point: {np.round(grad, 3).tolist()}')

    nm.save(models, meta, MODELS_DIR / f'function_{n}_nn.pt')
    return meta


## Train all 8 functions

In [3]:
all_meta = {}
for n in range(1, 9):
    print('=' * 72)
    all_meta[n] = search_and_save(n, verbose=True)
    print()


F1: 22 pts, 2D, baseline RMSE = 0.0016
  All configs (sorted):
    H= 16  ensemble    CV RMSE = 0.0022 ← BEST
    H= 16  dropout     CV RMSE = 0.0022
    H= 32  dropout     CV RMSE = 0.0023
    H= 16  wd          CV RMSE = 0.0026
    H= 32  ensemble    CV RMSE = 0.0026
    H= 16  plain       CV RMSE = 0.0027
    H= 32  wd          CV RMSE = 0.0031
    H= 32  plain       CV RMSE = 0.0036
  → best: H=16, ensemble, RMSE=0.0022 (-37.0% vs baseline) ✗ WORSE than baseline


  Gradient dY/dx at best point: [0.028999999165534973, -0.024000000208616257]



F2: 22 pts, 2D, baseline RMSE = 0.2413
  All configs (sorted):
    H= 32  dropout     CV RMSE = 0.1583 ← BEST
    H= 16  dropout     CV RMSE = 0.1655
    H= 16  ensemble    CV RMSE = 0.2324
    H= 16  plain       CV RMSE = 0.2522
    H= 16  wd          CV RMSE = 0.2543
    H= 32  ensemble    CV RMSE = 0.2554
    H= 32  wd          CV RMSE = 0.2812
    H= 32  plain       CV RMSE = 0.2827
  → best: H=32, dropout, RMSE=0.1583 (+34.4% vs baseline) ✓ beats baseline
  Gradient dY/dx at best point: [0.3919999897480011, 0.8080000281333923]



F3: 27 pts, 3D, baseline RMSE = 0.0732
  All configs (sorted):
    H= 32  ensemble    CV RMSE = 0.0714 ← BEST
    H= 16  ensemble    CV RMSE = 0.0733
    H= 16  dropout     CV RMSE = 0.0795
    H= 32  dropout     CV RMSE = 0.0808
    H= 32  wd          CV RMSE = 0.0810
    H= 32  plain       CV RMSE = 0.0836
    H= 16  wd          CV RMSE = 0.0855
    H= 16  plain       CV RMSE = 0.0892
  → best: H=32, ensemble, RMSE=0.0714 (+2.4% vs baseline) ✓ beats baseline


  Gradient dY/dx at best point: [0.03500000014901161, -0.08500000089406967, -0.013000000268220901]



...

  Gradient dY/dx at best point: [-0.3190000057220459, -0.6990000009536743, 0.19900000095367432, -0.10300000011920929, -0.878000020980835]



F7: 42 pts, 6D, baseline RMSE = 0.7730
  All configs (sorted):
    H= 16  ensemble    CV RMSE = 0.3225 ← BEST
    H= 32  ensemble    CV RMSE = 0.3748
    H= 32  dropout     CV RMSE = 0.4239
    H= 32  wd          CV RMSE = 0.4272
    H= 32  plain       CV RMSE = 0.4379
    H= 16  wd          CV RMSE = 0.4488
    H= 16  plain       CV RMSE = 0.4572
    H= 16  dropout     CV RMSE = 0.5090
  → best: H=16, ensemble, RMSE=0.3225 (+58.3% vs baseline) ✓ beats baseline


  Gradient dY/dx at best point: [-0.8880000114440918, -3.239000082015991, 2.171999931335449, 0.029999999329447746, -3.871000051498413, 0.19200000166893005]



F8: 52 pts, 8D, baseline RMSE = 1.2009
  All configs (sorted):
    H= 32  ensemble    CV RMSE = 0.3694 ← BEST
    H= 32  wd          CV RMSE = 0.3748
    H= 16  ensemble    CV RMSE = 0.3786
    H= 32  plain       CV RMSE = 0.3834
    H= 16  dropout     CV RMSE = 0.4253
    H= 32  dropout     CV RMSE = 0.4340
    H= 16  wd          CV RMSE = 0.4682
    H= 16  plain       CV RMSE = 0.4713
  → best: H=32, ensemble, RMSE=0.3694 (+69.2% vs baseline) ✓ beats baseline


  Gradient dY/dx at best point: [0.054999999701976776, 0.17800000309944153, -0.34599998593330383, -0.004999999888241291, 0.2290000021457672, 0.013000000268220901, -0.27000001072883606, -0.2540000081062317]



## F1 sign classifier

F1 is numerically ~0 for almost all points. Training an NN classifier on sign(y > 0) gives the analyze step a map of "where is the function positive" — now especially useful since W10 confirmed F1 is deterministic (W3's 3.65e-7 is a real reproducible peak, not noise).


In [4]:
X1, Y1 = load(1)
pos_frac = (Y1 > 0).mean()
print(f'F1: {len(Y1)} pts, {(Y1 > 0).sum()} positive, {(Y1 <= 0).sum()} zero-or-negative ({pos_frac:.0%} positive)')

if 0 < pos_frac < 1:
    clf, loo_acc = nm.train_sign_classifier(X1, Y1, hidden=16)
    nm.save_classifier(clf, loo_acc, d_in=X1.shape[1], hidden=16, path=MODELS_DIR / 'function_1_classifier.pt')
    print(f'Sign classifier trained. LOO accuracy = {loo_acc:.2%}')
else:
    print('Classifier skipped (all samples are one class).')


F1: 22 pts, 15 positive, 7 zero-or-negative (68% positive)


Sign classifier trained. LOO accuracy = 63.64%


## Summary

In [5]:
print(f"{'F':>2}  {'H':>3}  {'variant':10s}  {'CV RMSE':>10s}  {'baseline':>10s}  {'improve%':>8s}  {'beats?':>6s}")
print('-' * 62)
for n, m in all_meta.items():
    improve = (m['baseline_rmse'] - m['cv_rmse']) / m['baseline_rmse'] * 100
    mark = '✓' if m['beats_baseline'] else '✗'
    print(f'{n:>2}  {m["hidden"]:>3}  {m["variant"]:10s}  {m["cv_rmse"]:>10.4f}  {m["baseline_rmse"]:>10.4f}  {improve:>+7.1f}%  {mark:>6s}')


 F    H  variant        CV RMSE    baseline  improve%  beats?
--------------------------------------------------------------
 1   16  ensemble        0.0022      0.0016    -37.0%       ✗
 2   32  dropout         0.1583      0.2413    +34.4%       ✓
 3   32  ensemble        0.0714      0.0732     +2.4%       ✓
 4   32  dropout         4.2914      9.8937    +56.6%       ✓
 5   32  plain         264.7497   2337.2301    +88.7%       ✓
 6   32  ensemble        0.2904      0.6826    +57.5%       ✓
 7   16  ensemble        0.3225      0.7730    +58.3%       ✓
 8   32  ensemble        0.3694      1.2009    +69.2%       ✓
